# A・B75・B50：航路延長ロールアウト（Windows）
学習済み30kモデルだけを使う追加評価です。学習・E/R・HBA/F・別のA条件は実行しません。
低CPAの同じ交差航路について、開始地点を初速×0/20/40秒だけ後方へ延長します。予測CPA距離を保ち、予測上の接近時刻を遅らせます。
既存main/additionalの結果は変更せず、独立した共通bank・保存先を作ります。

20/40秒は初速による予測時間の増分です。Referenceは加速するため実際の猶予は軌跡から確認します。交差領域・出口・航路方向を保ち、出発点を後方へ延ばすので、出発点を共通の正方形外周には揃えません。


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from tqdm.notebook import tqdm
from scripts.main_route_extension_runner import prepare, checkpoints, select_scenarios, evaluate_grid, status, export_results
from uav_safe_marl.evaluation import load_scenario_bank
from uav_safe_marl.runtime import runtime_manifest
SOURCE = ROOT / 'results' / 'main_additional_20261120'
OUTPUT = ROOT / 'results' / 'main_route_extension_20261127'
BANK_PER_N = 100
EVAL_PER_N = 10
COUNTS = (2, 4, 8, 16)
LEADS = (0., 20., 40.)
SEEDS = (0, 1, 2)
EVALUATION_DEVICE = 'cpu'
RUN_PROBE = True
RUN_FULL = True
RUN_EXPORT = True


## 共通bankの固定
BANK_PER_N=100で400元形状×3延長条件を最初に固定します。まずEVAL_PER_N=10の40元形状・120シナリオを評価し、3方式×3seedで1,080episode。
EVAL_PER_N=100に増やすと同じbankの10,800episodeへ拡張できます。完了caseは再実行しません。
延長条件間は同じ元形状なので、独立形状が120個あるとは数えません。
途中再開は同じ設定・OUTPUTで実行。BANK_PER_N/延長条件等を変える場合は別OUTPUTへ保存します。EVAL_PER_Nだけなら同じOUTPUTで増やせます。


In [ ]:
experiment = prepare(SOURCE, OUTPUT, per_n=BANK_PER_N, counts=COUNTS, leads=LEADS)
print(f"元形状: {experiment['base_geometries']} / 延長後シナリオ: {experiment['scenarios']}")
print(f"全量: {experiment['scenarios'] * 3 * len(SEEDS):,} episode / 最大step: {experiment['max_steps']:,}")
selected_ids = select_scenarios(OUTPUT, EVAL_PER_N)
print('今回の評価量:', len(selected_ids) * 3 * len(SEEDS), 'episode')
print('生成試行数:', experiment['generation_draws'])
print('保存先:', OUTPUT)


## 評価モデルとデバイス
既存の親実験A/B、追加実験B50、またはSOURCE/modelsの正規名モデルを利用します。不足時は必要なパスを表示して停止します。
CPU評価を明示しています。安全QPもCPUで動作します。CUDAへ変える場合は開始前に設定し、同じ保存先の途中で変更しないでください。


In [ ]:
models = checkpoints(OUTPUT, SEEDS)
for (method, seed), path in models.items(): print(f'{method}/{seed}: {path}')
print('選択した評価デバイス:', EVALUATION_DEVICE)
print(json.dumps(runtime_manifest(), ensure_ascii=False, indent=2))


## 短い実走確認
最小/最大N×最短/最長延長の共通4シナリオをseed0の3方式で実行し、保存・所要時間を確認します。
モデルの読み込み・実行ごとに方式とhashを検証。case進捗に加えて、飛行中の実step数を表示します。
完了caseは全量評価で再利用します。RUN_FULL=Falseにすればこの確認だけで止まります。


In [ ]:
bank = load_scenario_bank(OUTPUT / 'banks' / 'route_extension.json')
probe_ids = []
for n in sorted({min(COUNTS), max(COUNTS)}):
    for lead in sorted({min(LEADS), max(LEADS)}):
        probe_ids.append(next(s['scenario_id'] for s in bank if s['actual_agent_count'] == n and s['route_extension_seconds'] == lead))
if RUN_PROBE:
    probe = evaluate_grid(OUTPUT, seeds=[SEEDS[0]], device=EVALUATION_DEVICE, scenario_ids=probe_ids, progress_factory=tqdm)
    print('確認済みcase:', probe['completed_cases'])


## 全量評価
1条件ずつ直列で実行し、並列workerは使いません。学習・Replay保存は行いません。
中断時に飛行中だった1caseはやり直し、完了markerと生NPZが揃うcaseは再実行しません。


In [ ]:
if RUN_FULL:
    result = evaluate_grid(OUTPUT, seeds=SEEDS, device=EVALUATION_DEVICE, scenario_ids=selected_ids, progress_factory=tqdm)
    print('今回の完了:', result['requested_completed_cases'], '/', result['requested_cases'])
    print('全bankで完了:', result['completed_cases'], '/', result['expected_all_seeds_cases'])


## 途中集計・取り出し
このセルは途中停止後にも実行できます。全episodeのJSON/CSV、生NPZ、bank、初期条件、出典、モデルhashを保持します。
集計の違反率は記録点での判定です。後から区間内最近接・接近ペア分布・Actor急変・可解性・通信離脱も解析できます。
成功限定の経路比/完了時間は、timeoutを含むepisode時間と分けて集計します。


In [ ]:
partial = status(OUTPUT)
print(json.dumps(partial, ensure_ascii=False, indent=2))


## Macへ持ち帰るzip
既定でA/B75/B50の評価用モデルも含めます（Replay・再開用checkpoint・runtime一時ファイルは除外）。
zip内のexport_manifest.jsonに全ファイルのhashとbyte数を記録します。保存先は評価フォルダの外です。
このセルだけ途中で実行すれば、その時点の完了結果を取り出せます。


In [ ]:
if RUN_EXPORT:
    archive = export_results(OUTPUT, include_models=True)
    print('持ち帰りzip:', archive)
    print(f'サイズ: {archive.stat().st_size / 1024**2:.1f} MiB')
